<a href="https://colab.research.google.com/github/nishalahmedpk/graphrag-timeseries/blob/main/mimic_iv_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-cloud-bigquery pandas pyarrow

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

Authenticated


In [ ]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "mimic-iv-research-491611"

client = bigquery.Client(project=PROJECT_ID)

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query on BigQuery and return a pandas DataFrame."""
    job = client.query(sql)
    return job.to_dataframe()

In [ ]:
df_test = run_query("""
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.patients`
LIMIT 10;
""")
df_test

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10078138,F,18,2110,2017 - 2019,NaT
1,10180372,M,18,2110,2008 - 2010,NaT
2,10686175,M,18,2110,2011 - 2013,NaT
3,10851602,F,18,2110,2014 - 2016,NaT
4,10902424,F,18,2110,2017 - 2019,NaT
5,11092326,M,18,2110,2008 - 2010,NaT
6,11289691,F,18,2110,2017 - 2019,NaT
7,11595073,M,18,2110,2011 - 2013,NaT
8,11739764,F,18,2110,2017 - 2019,NaT
9,11776346,F,18,2110,2008 - 2010,NaT


In [ ]:
def summarize_df(df: pd.DataFrame, name: str = "table"):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("\nmissing (top 10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    print("\ndtypes:")
    print(df.dtypes)

In [ ]:
def get_core_stay_context(stay_id: int) -> pd.DataFrame:
    sql = f"""
    SELECT
      i.stay_id,
      i.subject_id,
      i.hadm_id,
      i.first_careunit,
      i.last_careunit,
      i.intime,
      i.outtime,
      p.gender,
      p.anchor_age,
      p.anchor_year_group,
      a.admission_type,
      a.insurance,
      a.language,
      a.marital_status,
      a.race
    FROM `physionet-data.mimiciv_3_1_icu.icustays` AS i
    JOIN `physionet-data.mimiciv_3_1_hosp.patients` AS p
      ON i.subject_id = p.subject_id
    JOIN `physionet-data.mimiciv_3_1_hosp.admissions` AS a
      ON i.hadm_id = a.hadm_id
    WHERE i.stay_id = {stay_id}
    """
    df = run_query(sql)
    return df

In [ ]:
# Get one random stay_id to play with
one_stay = run_query("""
SELECT stay_id
FROM `physionet-data.mimiciv_3_1_icu.icustays`
ORDER BY RAND()
LIMIT 1;
""")
stay_id = int(one_stay["stay_id"].iloc[0])
stay_id

32423515

In [ ]:
core = get_core_stay_context(stay_id)
summarize_df(core, "core_stay_context")
core


=== core_stay_context ===
shape: (1, 15)
columns: ['stay_id', 'subject_id', 'hadm_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'gender', 'anchor_age', 'anchor_year_group', 'admission_type', 'insurance', 'language', 'marital_status', 'race']

missing (top 10):
stay_id              0
subject_id           0
hadm_id              0
first_careunit       0
last_careunit        0
intime               0
outtime              0
gender               0
anchor_age           0
anchor_year_group    0
dtype: int64

dtypes:
stay_id                       Int64
subject_id                    Int64
hadm_id                       Int64
first_careunit               object
last_careunit                object
intime               datetime64[us]
outtime              datetime64[us]
gender                       object
anchor_age                    Int64
anchor_year_group            object
admission_type               object
insurance                    object
language                     object
mar

,stay_id,subject_id,hadm_id,first_careunit,last_careunit,intime,outtime,gender,anchor_age,anchor_year_group,admission_type,insurance,language,marital_status,race
0,32423515,16413804,25124364,Neuro Stepdown,Neuro Stepdown,2162-12-30 23:57:41,2163-01-05 03:15:22,M,71,2017 - 2019,EW EMER.,Medicare,English,MARRIED,WHITE


In [ ]:
def get_stay_labs(stay_id: int, hours: int = 24) -> pd.DataFrame:
    sql = f"""
    WITH stay AS (
      SELECT stay_id, subject_id, hadm_id, intime
      FROM `physionet-data.mimiciv_3_1_icu.icustays`
      WHERE stay_id = {stay_id}
    )
    SELECT
      l.subject_id,
      l.hadm_id,
      l.itemid,
      l.charttime,
      l.valuenum,
      l.valueuom
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` AS l
    JOIN stay AS s
      ON l.hadm_id = s.hadm_id
    WHERE l.charttime BETWEEN s.intime
          AND DATETIME_ADD(s.intime, INTERVAL {hours} HOUR)
      AND l.valuenum IS NOT NULL
    ORDER BY l.charttime
    """
    df = run_query(sql)
    return df

In [ ]:
labs_24h = get_stay_labs(stay_id, hours=24)
summarize_df(labs_24h, "labs_24h")
labs_24h.head()


=== labs_24h ===
shape: (27, 6)
columns: ['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum', 'valueuom']

missing (top 10):
valueuom      4
subject_id    0
hadm_id       0
itemid        0
charttime     0
valuenum      0
dtype: int64

dtypes:
subject_id             Int64
hadm_id                Int64
itemid                 Int64
charttime     datetime64[us]
valuenum             float64
valueuom              object
dtype: object


,subject_id,hadm_id,itemid,charttime,valuenum,valueuom
0,16413804,25124364,51248,2162-12-31 08:01:00,29.2,pg
1,16413804,25124364,51249,2162-12-31 08:01:00,31.7,g/dL
2,16413804,25124364,52172,2162-12-31 08:01:00,47.8,fL
3,16413804,25124364,51250,2162-12-31 08:01:00,92.0,fL
4,16413804,25124364,50947,2162-12-31 08:01:00,1.0,None


In [ ]:
def get_stay_vitals(stay_id: int, hours: int = 24) -> pd.DataFrame:
    sql = f"""
    WITH stay AS (
      SELECT stay_id, intime
      FROM `physionet-data.mimiciv_3_1_icu.icustays`
      WHERE stay_id = {stay_id}
    )
    SELECT
      c.stay_id,
      c.itemid,
      c.charttime,
      c.valuenum,
      c.valueuom
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` AS c
    JOIN stay AS s
      ON c.stay_id = s.stay_id
    WHERE c.charttime BETWEEN s.intime
          AND DATETIME_ADD(s.intime, INTERVAL {hours} HOUR)
      AND c.valuenum IS NOT NULL
    ORDER BY c.charttime
    """
    df = run_query(sql)
    return df

In [ ]:
vitals_24h = get_stay_vitals(stay_id, hours=24)
summarize_df(vitals_24h, "vitals_24h")
vitals_24h.head()


=== vitals_24h ===
shape: (438, 5)
columns: ['stay_id', 'itemid', 'charttime', 'valuenum', 'valueuom']

missing (top 10):
valueuom     255
stay_id        0
itemid         0
charttime      0
valuenum       0
dtype: int64

dtypes:
stay_id               Int64
itemid                Int64
charttime    datetime64[us]
valuenum            float64
valueuom             object
dtype: object


,stay_id,itemid,charttime,valuenum,valueuom
0,32423515,220210,2162-12-31 01:57:00,19.0,insp/min
1,32423515,220045,2162-12-31 01:58:00,70.0,bpm
2,32423515,220277,2162-12-31 01:58:00,96.0,%
3,32423515,223769,2162-12-31 02:00:00,100.0,%
4,32423515,229321,2162-12-31 02:00:00,2.0,None


In [ ]:
def get_admission_diagnoses(hadm_id: int) -> pd.DataFrame:
    sql = f"""
    SELECT d.*
    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` AS d
    WHERE d.hadm_id = {hadm_id}
    """
    return run_query(sql)

In [ ]:
hadm_id = int(core["hadm_id"].iloc[0])
diagnoses = get_admission_diagnoses(hadm_id)
summarize_df(diagnoses, "diagnoses")
diagnoses.head()


=== diagnoses ===
shape: (13, 5)
columns: ['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']

missing (top 10):
subject_id     0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
dtype: int64

dtypes:
subject_id      Int64
hadm_id         Int64
seq_num         Int64
icd_code       object
icd_version     Int64
dtype: object


,subject_id,hadm_id,seq_num,icd_code,icd_version
0,16413804,25124364,1,S065X0A,10
1,16413804,25124364,2,G935,10
2,16413804,25124364,3,I248,10
3,16413804,25124364,4,G8194,10
4,16413804,25124364,5,G629,10


In [ ]:
def get_stay_bundle(stay_id: int, hours: int = 24):
    core = get_core_stay_context(stay_id)
    if core.empty:
        raise ValueError(f"stay_id {stay_id} not found")

    hadm_id = int(core["hadm_id"].iloc[0])

    labs = get_stay_labs(stay_id, hours=hours)
    vitals = get_stay_vitals(stay_id, hours=hours)
    diagnoses = get_admission_diagnoses(hadm_id)

    print(f"=== ICU stay {stay_id} (hadm_id={hadm_id}) ===")
    summarize_df(core, "core")
    summarize_df(labs, "labs")
    summarize_df(vitals, "vitals")
    summarize_df(diagnoses, "diagnoses")

    return {
        "core": core,
        "labs": labs,
        "vitals": vitals,
        "diagnoses": diagnoses,
    }

In [ ]:
bundle = get_stay_bundle(stay_id, hours=24)
bundle["core"]

=== ICU stay 32423515 (hadm_id=25124364) ===

=== core ===
shape: (1, 15)
columns: ['stay_id', 'subject_id', 'hadm_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'gender', 'anchor_age', 'anchor_year_group', 'admission_type', 'insurance', 'language', 'marital_status', 'race']

missing (top 10):
stay_id              0
subject_id           0
hadm_id              0
first_careunit       0
last_careunit        0
intime               0
outtime              0
gender               0
anchor_age           0
anchor_year_group    0
dtype: int64

dtypes:
stay_id                       Int64
subject_id                    Int64
hadm_id                       Int64
first_careunit               object
last_careunit                object
intime               datetime64[us]
outtime              datetime64[us]
gender                       object
anchor_age                    Int64
anchor_year_group            object
admission_type               object
insurance                    object
languag

,stay_id,subject_id,hadm_id,first_careunit,last_careunit,intime,outtime,gender,anchor_age,anchor_year_group,admission_type,insurance,language,marital_status,race
0,32423515,16413804,25124364,Neuro Stepdown,Neuro Stepdown,2162-12-30 23:57:41,2163-01-05 03:15:22,M,71,2017 - 2019,EW EMER.,Medicare,English,MARRIED,WHITE
